In [1]:
!pip install catboost

  Using cached contourpy-1.3.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (118 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 4.4 MB/s  0:00:22 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 1.9 MB/s  0:00:05 1.9 MB/s eta 0:00:01
Using cached contourpy-1.3.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (362 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (5.0 MB)
Using cached kiwisolver-1.5.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.5 MB)
   ━━━

In [1]:
import os

os.chdir("/home/joshbeckley/Documents/GitHub/SES-Hackathon-2026")

print(os.getcwd())

/home/joshbeckley/Documents/GitHub/SES-Hackathon-2026


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from statsmodels.tsa.ar_model import AutoReg


# ======================
# CONFIG
# ======================

FILE = "notebooks/winning_dataset.csv"
TARGET = "usd_zar_28"
DATE = "date"

TRAIN_END = "2021-01-01"


# ======================
# LOAD DATA
# ======================

df = pd.read_csv(FILE)

df[DATE] = pd.to_datetime(df[DATE])
df = df.sort_values(DATE)




FileNotFoundError: [Errno 2] No such file or directory: 'notebooks/winning_dataset.csv'

In [ ]:
# ======================
# FEATURE ENGINEERING
# ======================

num_cols = df.select_dtypes("number").columns.drop(TARGET)


for c in num_cols:

    for lag in [1, 2, 5, 10, 20, 28]:
        df[f"{c}_lag{lag}"] = df[c].shift(lag)

    for w in [5, 10, 20]:
        df[f"{c}_ma{w}"] = df[c].rolling(w).mean()
        df[f"{c}_std{w}"] = df[c].rolling(w).std()

    df[f"{c}_return"] = df[c].pct_change()


df = df.dropna()


# ======================
# SPLIT
# ======================

train = df[df[DATE] < TRAIN_END]
test = df[df[DATE] >= TRAIN_END]


features = [
    c for c in df.columns
    if c not in [DATE, TARGET]
]


X_train = train[features]
X_test = test[features]

y_train = train[TARGET]
y_test = test[TARGET]


print("Train:", train.shape)
print("Test :", test.shape)


# ======================
# RMSE FUNCTION
# ======================

def rmse(name, pred):
    score = np.sqrt(mean_squared_error(y_test, pred))
    results[name] = score
    print(f"{name:<20} {score:.6f}")


results = {}
preds = {}


# ======================
# BASELINES
# ======================

print("\nBASELINES")

rmse(
    "Random Walk",
    test["usd_zar"]
)


ar = AutoReg(
    train["usd_zar"],
    lags=1
).fit()


ar_pred = ar.predict(
    start=len(train),
    end=len(train)+len(test)-1
)


rmse(
    "AR(1)",
    ar_pred
)


# ======================
# MODELS
# ======================

models = {

    "Ridge": Ridge(alpha=10),

    "Random Forest": RandomForestRegressor(
        n_estimators=400,
        max_depth=10,
        n_jobs=-1,
        random_state=1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=400,
        max_depth=10,
        n_jobs=-1,
        random_state=1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        num_leaves=32,
        random_state=1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=1000,
        learning_rate=0.02,
        depth=6,
        verbose=False
    )
}


print("\nMODELS")


for name, model in models.items():

    print("Training", name)

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    preds[name] = pred

    rmse(
        name,
        pred
    )


# ======================
# ENSEMBLE
# ======================

ensemble = np.mean(
    [
        preds["XGBoost"],
        preds["LightGBM"],
        preds["CatBoost"],
        preds["Extra Trees"]
    ],
    axis=0
)


rmse(
    "Ensemble",
    ensemble
)


# ======================
# RESULTS
# ======================

print("\nRESULTS")

results = (
    pd.Series(results)
    .sort_values()
)


print(results)


results.to_csv(
    "rmse_results.csv"
)


# ======================
# FEATURE IMPORTANCE
# ======================

model = models["LightGBM"]

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})


importance = importance.sort_values(
    "importance",
    ascending=False
)


importance.to_csv(
    "feature_importance.csv",
    index=False
)


print("\nTop features:")
print(importance.head(20))